In [1]:
import warnings
warnings.filterwarnings('ignore')

# 데이터 처리 및 임베딩 기법

# 라이브러리 설치

## 설치하는 라이브러리의 역할  

데이터 추출 및 전처리(parsing)  
`pypdf`: PDF 파일에서 텍스트를 읽고 페이지를 분할하거나 합치는 등 PDF 데이터를 다루는 라이브러리  
`bs4(BeautifulSoup4)`: HTML이나 XML 파일(웹 페이지)에서 원하는 데이터를 쉽게 추출하는 웹 크롤링(스크레이핑) 라이브러리  
`jq`: 복잡한 JSON 형식의 데이터를 필터링하고 구조화하는 데 특화된 라이브러리  

자연어 처리 및 토큰화(NLP & Tokenization)  
`tiktoken`: OpenAI 모델이 텍스트를 처리하는 단위인 '토큰'으로 나누는 속도가 매우 빠른 토크나이저 라이브러리  
`transformers`: `Hugging Face`에서 제공하는 라이브러리로 BERT, GPT, Llama 등 많은 최신 AI 모델을 불러오는 라이브러리  
`langdetect`: 입력된 텍스트가 한국어인지, 영어인지 등 언어를 자동으로 감지해주는 라이브러리  

LangChain 생태계  
`langchain_experimental`: LangChain의 새로운 기능이나 실험적인 코드가 담겨있는 라이브러리  
`langchain_huggingface`: `Hugging Face`에 올라온 모델들을 LangChain 프레임워크 내에서 쉽게 쓸 수 있도록 연결하는 라이브러리  
`langchain_ollama`: 로컬 PC에서 AI돌리는 `Ollama`를 LangChain과 연동하여 오프라인 AI 서비스를 만들 때 사용하는 라이브러리  

기타 유틸리디  
`deepl`: DeepL 번역기 API를 파이썬에서 직접 호출하여 고성능 기계 번역을 수행하는 라이브러리

In [2]:
# !pip install pypdf bs4 jq tiktoken transformers langdetect langchain_experimental langchain_huggingface langchain_ollama deepl

# 환경 설정

## .env 환경 변수

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

## 기본 라이브러리

In [4]:
import os, json
from glob import glob
from pprint import pprint
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 다양한 형식의 문서 처리하기

## PDF 문서 가져오기

PDF 파일에서 텍스트를 추출(페이지별로 구분하여)을 읽어들이기(문서 객체로 변환) 위해서 PyPDFLoader를 import 한다.  
pypdf 라이브러리가 설치되어있어야 정상적으로 동작한다.

In [5]:
from langchain_community.document_loaders import PyPDFLoader

In [6]:
# PyPDFLoader 클래스의 생성자로 읽어들일 PDF 파일의 경로와 이름을 넘겨서 PyPDFLoader 클래스 객체를 생성한다.
pdf_loader = PyPDFLoader(file_path='./data/transformer.pdf')

# PyPDFLoader 객체에서 load() 메소드를 실행해서 실제 PDF 파일에서 텍스트를 불러온다.
# 이때, PDF 파일의 각 페이지를 텍스트로 추출하여 Document 객체가 저장된 리스트 형태로 불러오고 각 Document 객체 내부에는 page_content(페이지 내부의 텍스트)와
# metadata(파일 이름, 페이지 번호 등) 정보가 저장되어 있다.
pdf_docs = pdf_loader.load()
print(len(pdf_docs))
print(type(pdf_docs))
print(pdf_docs)

15
<class 'list'>
[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/transformer.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kais

In [7]:
pdf_docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/transformer.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\n

In [8]:
pdf_docs[0].metadata

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2024-04-10T21:11:43+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2024-04-10T21:11:43+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': './data/transformer.pdf',
 'total_pages': 15,
 'page': 0,
 'page_label': '1'}

In [9]:
pdf_docs[0].page_content

'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence

## 웹 문서 가져오기

사용자 에이전트(USER_AGENT) 식별 정보를 설정한다.  
USER_AGENT는 웹 브라우저나 프로그램을 통해 웹사이트에 접속할 때 '내가 어떤 프로그램으로 접속했는지' 웹 서버에 알려주는 식별 정보이다.  
최근 많은 웹사이트 API 서버는 자동화된 크롤러, 봇의 접근을 방지하거나 보안 정책에 의해 USER_AGENT 헤더가 비어있는 요청을 차단하기 때문에 'MyLLMApp/1.0'와 같이 임의의 이름, 버전 문자열을 지정함으로써 웹 서버에 요청시 차단되지 않고 정상적으로 데이터를 로드할 수 있도록 안전 장치를 마련한 것이다.

In [10]:
os.environ['USER_AGENT'] = 'MyLLMApp/1.0'

특정 웹 페이지의 내용을 읽어 텍스트를 추출해서 읽어들이기(문서 객체로 변환) 위해서 WebBaseLoader를 import 한다.  
bs4 라이브러리가 설치되어있어야 정상적으로 동작한다.

In [11]:
from langchain_community.document_loaders import WebBaseLoader

In [12]:
# WebBaseLoader 클래스의 생성자로 읽어들일 웹 문서의 주소를 넘겨서 WebBaseLoader 클래스 객체를 생성한다.
web_loader = WebBaseLoader(['https://python.langchain.com/', 'https://js.langchain.com/'])

# WebBaseLoader 객체에서 load() 메소드를 실행해서 실제 웹 문서의 텍스트 읽어온다.
# 이때, 웹 문서의 텍스트를 추출하여 Document 객체가 저장된 리스트 형태로 불러오고 각 Document 객체 내부에는 page_content(웹 문서의 텍스트)와 
# metadata(주소, 타이틀, 설명 등) 정보가 저장되어 있다.
web_docs = web_loader.load()
print(len(web_docs))
print(type(web_docs))
print(web_docs)

2
<class 'list'>
[Document(metadata={'source': 'https://python.langchain.com/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMess

In [13]:
web_docs[0]

Document(metadata={'source': 'https://python.langchain.com/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-ter

In [14]:
web_docs[0].metadata

{'source': 'https://python.langchain.com/',
 'title': 'LangChain overview - Docs by LangChain',
 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.',
 'language': 'en'}

In [15]:
web_docs[0].page_content

'LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringMCPHuman-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIProductionDeploymentObservabilityOn 

## JSON 문서 가져오기

JSON 또는 JSONL 파일의 내용을 읽어서 구조를 분석한 후 텍스트를 추출해서 읽어들이기(문서 객체로 변환) 위해서 JSONLoader를 import 한다.  
jq 라이브러리가 설치되어있어야 정상적으로 동작한다.

In [16]:
from langchain_community.document_loaders import JSONLoader

In [17]:
# JSONLoader 클래스의 생성자로 읽어들일 JSON 파일의 경로와 이름, JSON 파일의 데이터 구조를 넘겨서 JSONLoader 클래스 객체를 생성한다.
json_loader = JSONLoader(
    # 읽어들일 JSON 파일의 경로와 파일 이름을 지정한다.
    file_path='./data/kakao_chat.json',
    # jq 라이브러리를 사용해서 전체 데이터에서 어디를 읽을지 지정한다.
    # '전체 데이터의 messages라는 리스트([])' 안의 딕셔너리에서 'content'라는 key에 할당된 value만 가져오라는 의미이다. 대화 내용만 추출한다.
    jq_schema='.messages[].content',
    # text_content 속성은 추출한 데이터를 단순 문자열로 취급 여부를 설정한다.
    # 기본값(False)을 사용하면 jq_schema를 지정해서 추출한 데이터를 JSON 데이터로 취급한다.
    # True를 사용하면 jq_schema로 추출된 데이터가 '단순 문자열' 형태일 때 추가 가공 없이 원본 문자열 그대로 사용하게 한다. 단순 문자열로 취급한다.
    text_content=True,
)

# JSONLoader 객체에서 load() 메소드를 실행해서 JSON 문서의 텍스트 읽어온다.
# 이때, JSON 파일에서 jq_schema로 지정한 텍스트를 추출하여 Document 객체가 저장된 리스트 형태로 불러오고 각 Document 객체 내부에는 page_content(JSON 문서의 
# 텍스트)와 metadata(JSON 파일 경로, 일련 번호) 정보가 저장되어 있다.
json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 2}, page_content='네, 안녕하세요. 오후 2시에 하기로 했어요.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 3}, page_content='확인했습니다. 회의실은 어디인가요?'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 4}, page_content='3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 5}, page_content='네, 모두 준비했습니다. 회의 때 뵙겠습니다 :)')]


In [18]:
json_docs[0]

Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.')

In [19]:
json_docs[0].metadata

{'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json',
 'seq_num': 1}

In [20]:
json_docs[0].page_content

'안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'

이전 코드에서 jq_schema와 text_content를 수정했다.

In [21]:
json_loader = JSONLoader(
    file_path='./data/kakao_chat.json',
    # 이전 코드는 '.messages[].content'였지만 '.content'를 제거했다.
    # '.messages[].content'는 messages 리스트의 딕셔너리에서 'content'라는 key에 할당된 value를 가져오라는 의미였지만 '.content'를 제거하면 messages 리스트의
    # 모든 내용(sender, timestamp, content)을 한 덩어리로 가져온다.
    jq_schema='.messages[]',
    # 가져올 데이터가 단순 문자열이 아니다. JSON 형태의 데이터이다.
    text_content=False,
)

# 각 Document 객체의 'page_content'에는 단순 메시지만 들어가는 것이 아니라 {"sender": "홍길동", "timestamp": ..., "content": ...}와 같은 JSON 구조 전체가
# 텍스트로 들어간다.
json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, page_content='{"sender": "\\uae40\\ucca0\\uc218", "timestamp": "2023-09-15 09:30:22", "content": "\\uc548\\ub155\\ud558\\uc138\\uc694 \\uc5ec\\ub7ec\\ubd84, \\uc624\\ub298 \\ud68c\\uc758 \\uc2dc\\uac04 \\ud655\\uc778\\ucc28 \\uc5f0\\ub77d\\ub4dc\\ub9bd\\ub2c8\\ub2e4."}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 2}, page_content='{"sender": "\\uc774\\uc601\\ud76c", "timestamp": "2023-09-15 09:31:05", "content": "\\ub124, \\uc548\\ub155\\ud558\\uc138\\uc694. \\uc624\\ud6c4 2\\uc2dc\\uc5d0 \\ud558\\uae30\\ub85c \\ud588\\uc5b4\\uc694."}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 3}, page_content='{"sender": "\\ubc15\\ubbfc\\uc218", "timestamp": "2023-09-15 09:32:18", "content": "\\ud655\\uc778\\ud588\\uc2b5\\ub2c8\\ub2e4. 

text_content 속성의 속성값을 False로 지정해서 읽어들이면 한글이 깨져(유니코드 문자로) 보이는 현상이 발생된다.

In [22]:
# 유니코드로 인코딩된 데이터들을 다시 한글로 디코딩된 데이터로 저장할 리스트를 선언한다.
decoded_json_docs = []

for doc in json_docs:
    # print(type(doc))
    # page_content에 문자열 형태로 들어있는 JSON 데이터를 파이썬의 딕셔너리 형태로 변환한다.
    # json 라이브러리의 loads() 메소드로 파이썬의 딕셔너리 형태로 바꿔주면 유니코드 형태로 보이던 문자열이 실제 한글로 해석된다.
    decoded_data = json.loads(doc.page_content)
    # print(type(decoded_data))
    # print(decoded_data)
    
    decoded_json_docs.append({
        # 기존의 metadata는 그대로 유지한다.
        'metadata': doc.metadata,
        # 유니코드가 한글로 해석된 문자열을 추가한다.
        # 'page_content': decoded_data
        # 예전 버전으로 작성된 코드는 아래와 같은 코드를 사용하는 경우가 있다.
        # json 라이브러리의 dumps() 메소드의 ensure_ascii 속성의 속성값을 False로 지정해야 유니코드가 한글로 해석된 문자열이 추가된다.
        'page_content': json.dumps(decoded_data, ensure_ascii=False)
    })
    
print(len(decoded_json_docs))
print(type(decoded_json_docs))
print(decoded_json_docs)

5
<class 'list'>
[{'metadata': {'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, 'page_content': '{"sender": "김철수", "timestamp": "2023-09-15 09:30:22", "content": "안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다."}'}, {'metadata': {'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 2}, 'page_content': '{"sender": "이영희", "timestamp": "2023-09-15 09:31:05", "content": "네, 안녕하세요. 오후 2시에 하기로 했어요."}'}, {'metadata': {'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 3}, 'page_content': '{"sender": "박민수", "timestamp": "2023-09-15 09:32:18", "content": "확인했습니다. 회의실은 어디인가요?"}'}, {'metadata': {'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 4}, 'page_content': '{"sender": "이영희", "timestamp": "2023-09-15 09:33:40", "content": "3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!"}'}, {'metadata': {'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\

텍스트 내용(page_content)과 부가 정보(metadata)를 하나로 묶어주는 LangChain의 표준 데이터 규격인 Document를 사용하기 위해 import 한다.  
딕셔너리 형태로 만들어준 데이터를 LangChain의 다른 기능(텍스트 분할, 벡터 저장 등)과 호환되도록 LangChain의 표준 데이터 규격인 Document 객체로 변환한다.

In [23]:
from langchain_core.documents import Document

In [24]:
# 딕셔너리 형태의 데이터를 Document 클래스 객체로 변환한 데이터로 저장할 리스트를 선언한다.
decoded_json_docs = []

for doc in json_docs:
    decoded_data = json.loads(doc.page_content)
    # 유니코드를 한글로 디코딩한 새로눈 Document 클래스 객체를 생성한다.
    # 한글이 유니코드 형태로 깨지지 않도록 ensure_ascii=False 속성을 설정해서, 한글로 디코딩된 내용의 문자열을 저장한다.
    document_obj = Document(metadata=doc.metadata, page_content=json.dumps(decoded_data, ensure_ascii=False))
    decoded_json_docs.append(document_obj)
    
print(len(decoded_json_docs))
print(type(decoded_json_docs))
print(decoded_json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, page_content='{"sender": "김철수", "timestamp": "2023-09-15 09:30:22", "content": "안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다."}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 2}, page_content='{"sender": "이영희", "timestamp": "2023-09-15 09:31:05", "content": "네, 안녕하세요. 오후 2시에 하기로 했어요."}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 3}, page_content='{"sender": "박민수", "timestamp": "2023-09-15 09:32:18", "content": "확인했습니다. 회의실은 어디인가요?"}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 4}, page_content='{"sender": "이영희", "timestamp": "2023-09-15 09:33:40", "content": "3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!"}'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\work

In [25]:
decoded_json_docs[0]

Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1}, page_content='{"sender": "김철수", "timestamp": "2023-09-15 09:30:22", "content": "안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다."}')

In [26]:
decoded_json_docs[0].metadata

{'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json',
 'seq_num': 1}

In [27]:
decoded_json_docs[0].page_content

'{"sender": "김철수", "timestamp": "2023-09-15 09:30:22", "content": "안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다."}'

JSON 파일 안의 특정 필드(`content`)를 본문(`page_content`)으로 지정하고 다른 필드(`sender`, `timestamp`)들을 추출해서 메타데이터(`metadata`)로 합친다.

각 JSON 레코드를 읽을 때 메타데이터를 어떻게 구성할지 정의하는 함수

In [28]:
# JSONLoader 객체의 metadata_func 속성에서 실행할 함수
def metadata_func(record, metadata):
    # print(record, metadata)
    # 현재 처리하는 JSON 파일에서 읽어들인 데이터 1건(딕셔녀리)에서 'sender'라는 key에 할당된 value를 얻어와서 메타데이터에 추가한다.
    metadata['sender'] = record['sender']
    # 현재 처리하는 JSON 파일에서 읽어들인 데이터 1건(딕셔녀리)에서 'timestamp'라는 key에 할당된 value를 얻어와서 메타데이터에 추가한다.
    metadata['timestamp'] = record['timestamp']
    return metadata

In [29]:
json_loader = JSONLoader(
    file_path='./data/kakao_chat.json',
    # 전체 JSON 데이터 중에서 'messages' 리스트를 구성하는 딕셔너리에서 'content'라는 key에 할당된 value를 얻어온다.
    # jq_schema='.messages[].content',
    # 전체 JSON 데이터 중에서 'messages' 리스트를 얻어온다.
    jq_schema='.messages[]',
    # jq_schema를 이용해서 얻어온 리스트를 구성하는 딕셔너리에서 'content'라는 key에 할당된 value를 얻어온다. 본문(page_content)으로 사용한다.
    content_key='content',
    # JSON 레코드를 읽을 때 메타데이터를 어떻게 구성할지 정의하는 함수를 실행한다. 메타 데이터(metadata)를 만든다.
    # metadata_func 속성으로 실행할 함수를 지정하면 함수의 첫 번째 인수로 JSON 파일에서 읽어들인 page_content 전체(데이터 1건, 레코드)가 넘어가고 두 번째
    # 인수로 두 번째 인수로 metadata가 자동으로 넘어간다.
    metadata_func=metadata_func,
)

json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1, 'sender': '김철수', 'timestamp': '2023-09-15 09:30:22'}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 2, 'sender': '이영희', 'timestamp': '2023-09-15 09:31:05'}, page_content='네, 안녕하세요. 오후 2시에 하기로 했어요.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 3, 'sender': '박민수', 'timestamp': '2023-09-15 09:32:18'}, page_content='확인했습니다. 회의실은 어디인가요?'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 4, 'sender': '이영희', 'timestamp': '2023-09-15 09:33:40'}, page_content='3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 5, 'sender': '김철수'

In [30]:
json_docs[0]

Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json', 'seq_num': 1, 'sender': '김철수', 'timestamp': '2023-09-15 09:30:22'}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.')

In [31]:
json_docs[0].metadata

{'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.json',
 'seq_num': 1,
 'sender': '김철수',
 'timestamp': '2023-09-15 09:30:22'}

In [32]:
json_docs[0].page_content

'안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'

일반적인 JSON 파일이 아니라, 한 줄에 하나의 JSON 객체가 저장된 JSONL(JSON Lines) 파일을 처리한다. 대용량 로그 데이터나 센서 데이터 또는 채팅 기록을 다룰 때 효율적인 방식이다.

In [33]:
json_loader = JSONLoader(
    file_path='./data/kakao_chat.jsonl',
    # 이전의 '.messages[]'와 달리 JSONL 파일은 한 줄이 하나의 레코드이므로 리스트를 순회할 필요 없이 바로 key 이름을 지정한다.
    jq_schema='.content',
    # json_lines 속성의 속성값을 True로 지정해서, JSONLoader에게 '한 줄 한 줄이 개별 JSON 데이터'라고 알려준다.
    # 이 속성이 있어야 JSONL 파일을 정상적으로 읽을 수 있다.
    json_lines=True,
)

json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 1}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 2}, page_content='네, 안녕하세요. 오후 2시에 하기로 했어요.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 3}, page_content='확인했습니다. 회의실은 어디인가요?'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 4}, page_content='3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 5}, page_content='네, 모두 준비했습니다. 회의 때 뵙겠습니다 :)')]


개별 줄 전체(`.`)를 대상으로 하되 그중 특정 키(`content`)만 본문(`page_content`)으로 불러오는 방식이다.

In [34]:
json_loader = JSONLoader(
    file_path='./data/kakao_chat.jsonl',
    # '.'은 현재 줄의 전체를 대상으로 하라는 의미이다.
    jq_schema='.',
    content_key='content',
    json_lines=True,
)

json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 1}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 2}, page_content='네, 안녕하세요. 오후 2시에 하기로 했어요.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 3}, page_content='확인했습니다. 회의실은 어디인가요?'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 4}, page_content='3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 5}, page_content='네, 모두 준비했습니다. 회의 때 뵙겠습니다 :)')]


이전의 JSON 파일을 읽어서 메타데이터를 추가했던 것 처럼 JSONL 파일을 읽어서 메타데이터를 추가한다.

In [35]:
json_loader = JSONLoader(
    file_path='./data/kakao_chat.jsonl',
    jq_schema='.',
    content_key='content',
    metadata_func=metadata_func,
    json_lines=True,
)

json_docs = json_loader.load()
print(len(json_docs))
print(type(json_docs))
print(json_docs)

5
<class 'list'>
[Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 1, 'sender': '김철수', 'timestamp': '2023-09-15 09:30:22'}, page_content='안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 2, 'sender': '이영희', 'timestamp': '2023-09-15 09:31:05'}, page_content='네, 안녕하세요. 오후 2시에 하기로 했어요.'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 3, 'sender': '박민수', 'timestamp': '2023-09-15 09:32:18'}, page_content='확인했습니다. 회의실은 어디인가요?'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 4, 'sender': '이영희', 'timestamp': '2023-09-15 09:33:40'}, page_content='3층 대회의실입니다. 프로젝트 진행 상황 정리해오시는 거 잊지 마세요!'), Document(metadata={'source': 'D:\\samsung_1\\python\\09_AI_RAG\\workspace\\data\\kakao_chat.jsonl', 'seq_num': 5, 'sender': 

## CSV 문서 가져오기

CSV 파일의 내용을 읽어들이기(문서 객체로 변환) 위해서 CSVLoader를 import 한다.  

from langchain_community.document_loaders.csv_loader import CSVLoader  
예전에는 langchain_community.document_loaders`.csv_loader` 모듈을 사용했지만 버전이 올라가면서 모든 로더는 langchain_community.document_loaders 모듈에서 일괄 관리되면서 아래와 같이 사용한다.

In [38]:
from langchain_community.document_loaders import CSVLoader

In [41]:
# CSVLoader 클래스의 생성자로 읽어들일 CSV 파일의 경로와 이름을 필요에 따라서 인코딩 방식을 넘겨서 CSVLoader 클래스 객체를 생성한다.
# UnicodeDecodeError: 'cp949' codec can't decode byte 0xed in position와 같은 에러가 발생되면 encoding='utf-8' 속성을 지정한다.
csv_loader = CSVLoader(file_path='./data/kbo_teams_2023.csv', encoding='utf-8')

# CSVLoader 객체에서 load() 메소드를 실행해서 CSV 문서의 텍스트 읽어온다.
# Document 객체가 저장된 리스트 형태로 불러오고 각 Document 객체 내부에는 page_content(CSV 파일의 텍스트)와 metadata(CSV 파일 경로, 줄 번호) 정보가 저장되어 
# 있다.
csv_docs = csv_loader.load()
print(len(csv_docs))
print(type(csv_docs))
print(csv_docs)

10
<class 'list'>
[Document(metadata={'source': './data/kbo_teams_2023.csv', 'row': 0}, page_content="Team: KIA 타이거즈\nCity: 광주\nFounded: 1982\nHome Stadium: 광주-기아 챔피언스 필드\nChampionships: 11\nIntroduction: KBO 리그의 전통 강호로, 역대 최다 우승 기록을 보유하고 있다. '타이거즈 스피릿'으로 유명하며, 양현종, 안치홍 등 스타 선수들을 배출했다. 광주를 연고로 하는 유일한 프로야구팀으로 지역 사랑이 강하다."), Document(metadata={'source': './data/kbo_teams_2023.csv', 'row': 1}, page_content='Team: 두산 베어스\nCity: 서울\nFounded: 1982\nHome Stadium: 잠실야구장\nChampionships: 6\nIntroduction: 2015년부터 2019년까지 5년 연속 한국시리즈에 진출한 강팀이다. 김태형 감독 체제에서 체계적인 선수 육성으로 주목받았으며, 정수빈, 김재환 등 핵심 선수들의 활약이 돋보인다.'), Document(metadata={'source': './data/kbo_teams_2023.csv', 'row': 2}, page_content="Team: SSG 랜더스\nCity: 인천\nFounded: 2000\nHome Stadium: 인천SSG랜더스필드\nChampionships: 5\nIntroduction: SK 와이번스에서 SSG 랜더스로 구단명을 변경했다. 2022년 와일드카드 결정전부터 한국시리즈까지 전승으로 우승을 차지하며 '노히트 행진'을 달성했다. 추신수, 김광현 등 스타 선수들이 포진해 있다."), Document(metadata={'source': './data/kbo_teams_2023.csv', 'row': 3}, page_content='Team: 삼성 라이온즈\nC

In [43]:
csv_docs[4]

Document(metadata={'source': './data/kbo_teams_2023.csv', 'row': 4}, page_content='Team: LG 트윈스\nCity: 서울\nFounded: 1982\nHome Stadium: 잠실야구장\nChampionships: 2\nIntroduction: 서울을 연고로 하는 인기 구단으로, 오랜 기간 우승이 없었으나 2023년 29년 만에 한국시리즈 우승을 차지했다. 김현수, 고우석 등 국가대표급 선수들이 포진해 있으며, 탄탄한 마운드가 강점이다.')

In [44]:
csv_docs[4].metadata

{'source': './data/kbo_teams_2023.csv', 'row': 4}

In [45]:
csv_docs[4].page_content

'Team: LG 트윈스\nCity: 서울\nFounded: 1982\nHome Stadium: 잠실야구장\nChampionships: 2\nIntroduction: 서울을 연고로 하는 인기 구단으로, 오랜 기간 우승이 없었으나 2023년 29년 만에 한국시리즈 우승을 차지했다. 김현수, 고우석 등 국가대표급 선수들이 포진해 있으며, 탄탄한 마운드가 강점이다.'

# 텍스트 분할 전략

## RecursiveCharacterTextSplitter 사용

LangChain에서 제공하는 고급 텍스트 분할 도구로 텍스트를 재귀적으로 분할하여 더 자연스러운 문서 조각(청크)를 생성한다.  
순차적 구분자 적용 순서: `\n\n`(문단, 단락 단위- 최우선) => `\n`(줄 단위) => `.`(문장 단위) => `' '`(공백, 단어 단위) => `''`(글자 단위 - 최후의 수단)  
CharacterTextSplitter보다 더 엄격하게 크기를 준수하려는 경향이 있다. => 01_LangChain의_주요_RAG_컴포넌트 예제에서 설명했다.

구분자를 지정해서(여러 개) 텍스트를 재귀적으로 분할하여 문서 조각(청크)을 생성하기 위해서 RecursiveCharacterTextSplitter를 import 한다.

In [47]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [61]:
# 문맥을 최대한 보존하면서 문서를 자르는 도구로 단순히 글자 수로만 자르는 것이 아니라 문단이나 줄바꿈 같은 구분자를 기준으로 자연스럽게 잘라준다.
# RecursiveCharacterTextSplitter 클래스의 생성자로 청크의 크기, 문맥을 유지하기 위해 겹치는 크기, 구분자, 길이를 재는 기준을 넘겨서 객체를 만든다.
text_splitter = RecursiveCharacterTextSplitter(
    # 텍스트를 재귀적으로 분할할 문서 조각(청크)의 크기를 1,000으로 지정한다.
    chunk_size=1000,
    # 텍스트를 문서 조각으로 자를때 문맥이 끊기는 것을 방지하기 위해서 앞의 문서 조각의 끝부분 내용을 뒤의 문서 조각이 포함해서 겹치는 크기를 200으로 지정한다.
    chunk_overlap=200,
    # \n\n => \n => '.' => ' ' => '' 순서로 텍스트를 문서 조각으로 자르는 단위를 지정한다. 여러개의 구분자를 지정할 수 있다.
    # 먼저 문단(\n\n) 단위로 잘라보고, 그래도 1,000자가 넘으면 줄바꿈(\n) 단위로 자른다. 이렇게 하면 문서 중간이 잘리는 현상을 최소화할 수 있다.
    separators=['\n\n', '\n'],
    # 문서 조각의 길이를 재는 함수는 지정한다. 필요에 따라 tiktoken을 사용해 토큰 수를 기준으로 잴 수도 있다.
    length_function=len,
)

# RecursiveCharacterTextSplitter 객체의 split_documents() 메소드를 이용해서 텍스트를 문서 조각으로 자른다.
texts = text_splitter.split_documents(pdf_docs)
print(len(texts))
print([len(text.page_content) for text in texts])

52
[984, 910, 975, 452, 930, 995, 902, 907, 994, 382, 923, 951, 216, 917, 996, 841, 988, 913, 905, 868, 928, 965, 943, 997, 196, 974, 971, 946, 930, 986, 943, 918, 734, 958, 946, 945, 617, 982, 988, 994, 624, 944, 909, 941, 914, 986, 925, 927, 847, 812, 815, 818]


In [71]:
# 각 문서 조각의 겹치는 부분을 확인한다.
for i in range(len(texts[:3])):
    print(texts[i].page_content[-200:])
    print('*' * 100)
    print(texts[i + 1].page_content[:200])
    print('=' * 100)
    print()

the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
****************************************************************************************************
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine transla

the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data.
****************************************************************************************************
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training dat

## CharacterTextSplitter와 정규 표현식 사용

정규 표현식을 사용하면 특정 패턴을 기반으로 텍스트를 더 정확하게 분할할 수 있어서 구조화된 텍스트나 특정 형식의 문서에 유용하다.  
제1조, 제1장, 문장 단위(마침표, 느낌표, 물음표)로 끝나는 문장에 활용한다.

기본적인 텍스트 분할기로 구분자를 지정해서(1개) 텍스트를 분할하여 문서 조각(청크)을 생성하기 위해서 CharacterTextSplitter를 import 한다.

In [72]:
from langchain_text_splitters import CharacterTextSplitter

텍스트를 단순히 글자 수로 자르는 것이 아니고 문장 부호(`.`, `!`, `?`)를 기준으로 의미있는 문장 단위로 정규 표현식을 사용해서 나눈다.

정규 표현식 `r'(?<=[.!?])\s+'`는 마침표(.), 물음표(?), 느낌표(!) 바로 뒤에 나오는 하나 이상의 공백 문자를 찾는 패턴이다.  
`(?<=...)`: 후방 탐색을 의미하고 조건(`[.!?]`)이 일치하는 위치를 찾되, 해당 문자를 매칭 결과에 포함시키지 않는 역할을 합니다.  
`[.!?]`: 마침표(.), 물음표(?), 느낌표(!) 중 하나를 의미한다.  
`\s+`: 공백 문자들, `\s`는 스페이스(공백), 탭(\t), 줄바꿈(\n) 등 모든 공백 문자를 의미하고 `+`는 직전의 문자가 1개 이상 연속하여 존재하는 경우를 뜻한다.

검색 조건으로만 사용되고 실제 치환이나 분할 시 잘려나가지 않습니다.

In [94]:
# CharacterTextSplitter 클래스의 생성자로 청크의 크기, 문맥을 유지하기 위해 겹치는 크기, 구분자를 넘겨서 객체를 만든다.
text_splitter = CharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    # 마침표, 느낌표, 물음표 중 하나가 나오고, 그 뒤에 공백이 이어지는 위치를 찾아서 나누는 구분자를 지정한다.
    separator=r'(?<=[.!?])\s+',
    # separator로 정규 표현식을 사용하려면 is_separator_regex 속성의 속성값을 True로 저정해야 한다.
    is_separator_regex=True,
    # 분할 기준이 된 문장 부호를 버리지 않고 문장 끝에 그대로 붙여둔다.
    keep_separator=True
)

texts = text_splitter.split_documents(json_docs)
print(len(texts))
print([len(text.page_content) for text in texts])

Created a chunk of size 11, which is longer than the specified 10
Created a chunk of size 13, which is longer than the specified 10


9
[31, 9, 15, 7, 11, 11, 27, 13, 13]


In [96]:
for i in range(len(texts[:3])):
    print(texts[i].page_content[-200:])
    print('*' * 100)
    print(texts[i + 1].page_content[:200])
    print('=' * 100)
    print()

안녕하세요 여러분, 오늘 회의 시간 확인차 연락드립니다.
****************************************************************************************************
네, 안녕하세요.

네, 안녕하세요.
****************************************************************************************************
오후 2시에 하기로 했어요.

오후 2시에 하기로 했어요.
****************************************************************************************************
확인했습니다.



## 토큰 수를 기반으로 분할

토큰 기반 분할은 LLM의 토큰 제한을 고려할 때 유용하며 각 문서 조각(청크)이 특정 토큰 수를 초과하지 않도록 조절 가능하다.  
tiktoken, transformers 라이브러리가 설치되어있어야 정상적으로 동작한다.

In [99]:
print(len(pdf_docs[0].page_content))
print(pdf_docs[0].page_content)

2857
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. 

tiktoken 라이브러리(OpenAI의 토큰 계산기)를 사용해 텍스트를 나누는 객체를 선언해서 일반적인 글자 수가 아니라, 실제 LLM이 받아들이는 토큰 수를 기준으로 자른다.

`cl100k_base`는 OpenAI가 개발하여 GPT-4, GPT-3.5-Turbo, text-embedding-3 등 널리 사용되는 대형 언어 모델(LLM) 및 임베딩 모델에 적용한 BPE(Byte Pair Encoding) 기반의 토크나이저 인코딩 방식이다.

In [115]:
# from_tiktoken_encoder() 메소드는 LLM의 토큰 수를 기준으로 텍스트를 측정 및 분할한다.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    # 토큰을 계산할 인코딩 방식(알고리즘)을 지정한다.
    encoding_name='o200k_base',
    # 특정 모델 이름을 직접 지정할 수도 있다.
    # model_name='gpt-4o-mini'를 사용하면 'encoding_name' 속성 설정을 생략해도 해당 모델에 맞게 자동으로 설정된다.
    # model_name='gpt-4o-mini',
    # 하나의 문서 조각(청크)에 들어갈 최대 문자수를 의미하는 것이 아니고 최대 토큰 수를 300개로 설정한다.
    # LLM은 한 번에 읽을 수 있는 토큰 양의 한계가 있으므로, 효율적인 정보 검색을 위해 적절한 크기(보통 300 ~ 1000)로 잘라준다.
    chunk_size=300,
    # 나눠진 청크들 사이에 중복 내용을 얼마나 둘지 결정한다. 보통 문맥 연결을 위해 20 ~ 50 정도를 주기도 한다.
    chunk_overlap=0,
)


chunks = text_splitter.split_documents(pdf_docs[:1])
print(len(chunks))
print([len(chunk.page_content) for chunk in chunks])

3
[1143, 1374, 338]
